<a href="https://www.kaggle.com/code/asadali15/airs-using-catboost-by-asad-ali?scriptVersionId=288189716" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [2]:
# ==========================================================
# 🎯 KAGGLE SUBMISSION: Train on Full Data + Test Predictions
# ==========================================================

import os
import numpy as np
import pandas as pd
import librosa
import scipy
import pickle
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_classif
from catboost import CatBoostClassifier
import seaborn as sns
import matplotlib.pyplot as plt

# =======================
# PATHS & PARAMETERS
# =======================
SOUND_DIR = "/kaggle/input/airs-ai-in-respiratory-sounds/sounds/sounds/"
TRAIN_PATH = "/kaggle/input/airs-ai-in-respiratory-sounds/train.csv"
TEST_PATH  = "/kaggle/input/airs-ai-in-respiratory-sounds/test.csv"
CACHE_FILE = "/kaggle/working/audio_cache_enhanced.pkl"

SR = 16000
DURATION = 2
TARGET_LEN = SR * DURATION
N_MFCC = 30

# =======================
# LOAD TRAINING DATA
# =======================
print("📂 Loading TRAINING data...")
df_train = pd.read_csv(TRAIN_PATH)

for col in df_train.columns:
    if df_train[col].isnull().sum() > 0:
        df_train[col].fillna(df_train[col].mode()[0], inplace=True)

print("📂 Loading TEST data...")
df_test = pd.read_csv(TEST_PATH)

for col in df_test.columns:
    if df_test[col].isnull().sum() > 0:
        df_test[col].fillna(df_test[col].mode()[0], inplace=True)

# =======================
# ENHANCED AUDIO FEATURE EXTRACTION
# =======================
def resample(y, orig_sr, target_sr=16000):
    if orig_sr != target_sr:
        y = scipy.signal.resample(y, int(len(y) * target_sr / orig_sr))
    return y

def pad_truncate(y, length):
    return np.pad(y, (0, max(0, length - len(y))))[:length]

def extract_audio(path):
    try:
        y, sr = librosa.load(path, sr=None)
        y = resample(y, sr, SR)
        y = pad_truncate(y, TARGET_LEN)

        mfcc = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=N_MFCC)
        delta = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)
        
        chroma = librosa.feature.chroma_stft(y=y, sr=SR)
        contrast = librosa.feature.spectral_contrast(y=y, sr=SR)
        tonnetz = librosa.feature.tonnetz(y=y, sr=SR)
        
        mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=128)
        tempogram = librosa.feature.tempogram(y=y, sr=SR)
        
        zcr = librosa.feature.zero_crossing_rate(y)
        
        spec_cent = librosa.feature.spectral_centroid(y=y, sr=SR)
        spec_rolloff = librosa.feature.spectral_rolloff(y=y, sr=SR)
        spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=SR)
        
        rmse = librosa.feature.rms(y=y)

        feats = np.concatenate([
            mfcc.mean(1), mfcc.std(1),
            delta.mean(1), delta.std(1),
            delta2.mean(1), delta2.std(1),
            chroma.mean(1), chroma.std(1),
            contrast.mean(1), contrast.std(1),
            tonnetz.mean(1),
            mel.mean(1), mel.std(1),
            [np.mean(tempogram)],
            zcr.mean(1), zcr.std(1),
            spec_cent.mean(1), spec_cent.std(1),
            spec_rolloff.mean(1), spec_rolloff.std(1),
            spec_bw.mean(1), spec_bw.std(1),
            rmse.mean(1), rmse.std(1)
        ])
        return feats
    except Exception as e:
        return np.zeros(30*2 + 30*2 + 30*2 + 12*2 + 7*2 + 6 + 128*2 + 1 + 1*2 + 1*2 + 1*2 + 1*2 + 1*2)

def build_audio(df, cache=None, is_test=False):
    if cache and os.path.exists(cache) and not is_test:
        print(f"📦 Loading cached features from {cache}...")
        with open(cache, "rb") as f:
            return pickle.load(f)

    print("🎵 Extracting ENHANCED audio features...")
    feats = [
        extract_audio(os.path.join(SOUND_DIR, f"{cid}.wav"))
        for cid in tqdm(df["candidateID"])
    ]

    audio_df = pd.DataFrame(feats)
    audio_df["candidateID"] = df["candidateID"].values

    if cache and not is_test:
        with open(cache, "wb") as f:
            pickle.dump(audio_df, f)

    return audio_df

# Extract audio features
train_audio = build_audio(df_train, CACHE_FILE, is_test=False)
test_audio = build_audio(df_test, is_test=True)

# =======================
# LABEL ENCODING FOR TRAINING DATA
# =======================
categorical_cols = [
    'gender','tbContactHistory','wheezingHistory',
    'phlegmCough','familyAsthmaHistory',
    'feverHistory','coldPresent'
]

df_train_enc = df_train.copy()
for col in categorical_cols:
    le = LabelEncoder()
    df_train_enc[col] = le.fit_transform(df_train_enc[col].astype(str))

# =======================
# LABEL ENCODING FOR TEST DATA (using training encoders)
# =======================
df_test_enc = df_test.copy()
for col in categorical_cols:
    le = LabelEncoder()
    # Fit on training data, transform test data
    le.fit(df_train[col].astype(str))
    df_test_enc[col] = le.transform(df_test[col].astype(str))

# =======================
# MERGE FEATURES
# =======================
X_train = pd.concat([
    df_train_enc.drop(['candidateID','disease'], axis=1),
    train_audio.drop(['candidateID'], axis=1)
], axis=1)

X_test = pd.concat([
    df_test_enc.drop(['candidateID'], axis=1),
    test_audio.drop(['candidateID'], axis=1)
], axis=1)

y_train = LabelEncoder().fit_transform(df_train['disease'])

# =======================
# IMPUTE & SCALE (using training data statistics)
# =======================
imputer = SimpleImputer(strategy="most_frequent")
X_train_imputed = imputer.fit_transform(X_train.values)
X_test_imputed = imputer.transform(X_test.values)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# =======================
# FEATURE SELECTION (on full training data)
# =======================
print("\n🔍 Selecting best features...")
selector = SelectKBest(f_classif, k=450)  # Optimal from testing
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

print(f"✅ Final features: 500 → 450")
print(f"✅ Train Samples: {X_train_selected.shape[0]}")
print(f"✅ Test Samples : {X_test_selected.shape[0]}")

# =======================
# TRAIN CATBOOST ON FULL TRAINING DATA
# =======================
print("\n" + "="*70)
print("🎯 TRAINING CatBoost on FULL training data")
print("="*70)

model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.005,
    depth=5,
    l2_leaf_reg=50,
    subsample=0.7,
    bootstrap_type='Bernoulli',
    eval_metric='MultiClass',
    random_seed=42,
    verbose=100,
    border_count=256,
    colsample_bylevel=0.7,
    min_data_in_leaf=15,
    grow_policy='Lossguide'
)

model.fit(X_train_selected, y_train)

print("\n✅ Model training completed!")

# =======================
# PREDICTIONS ON TEST DATA
# =======================
y_test_pred = model.predict(X_test_selected)

print("\n" + "="*70)
print("📊 MAKING PREDICTIONS ON TEST DATA")
print("="*70)

# =======================
# CREATE SUBMISSION FILE
# =======================
le_disease = LabelEncoder()
le_disease.fit(df_train['disease'])

# Get disease names
disease_names = le_disease.classes_
predictions_names = le_disease.inverse_transform(y_test_pred)

# Create submission dataframe
submission_df = pd.DataFrame({
    'candidateID': df_test['candidateID'].values,
    'disease': predictions_names
})

print(f"\n✅ Predictions shape: {submission_df.shape}")
print(f"✅ Sample predictions:")
print(submission_df.head(10))

# Save submission file
submission_path = "/kaggle/working/submission.csv"
submission_df.to_csv(submission_path, index=False)

print(f"\n🎉 Submission file saved to: {submission_path}")
print(f"✅ Total predictions: {len(submission_df)}")

# Show disease distribution
print("\n" + "="*70)
print("📊 PREDICTION DISTRIBUTION")
print("="*70)
print(submission_df['disease'].value_counts())

print("\n✅ SUBMISSION READY FOR KAGGLE!")


📂 Loading TRAINING data...
📂 Loading TEST data...
🎵 Extracting ENHANCED audio features...


100%|██████████| 546/546 [00:06<00:00, 84.21it/s] 


🎵 Extracting ENHANCED audio features...


100%|██████████| 338/338 [00:00<00:00, 783.68it/s]



🔍 Selecting best features...
✅ Final features: 500 → 450
✅ Train Samples: 546
✅ Test Samples : 338

🎯 TRAINING CatBoost on FULL training data
0:	learn: 1.0969120	total: 68.2ms	remaining: 3m 24s
100:	learn: 0.9560355	total: 163ms	remaining: 4.69s
200:	learn: 0.8560761	total: 258ms	remaining: 3.59s
300:	learn: 0.7842680	total: 349ms	remaining: 3.13s
400:	learn: 0.7304876	total: 444ms	remaining: 2.88s
500:	learn: 0.6901423	total: 538ms	remaining: 2.68s
600:	learn: 0.6594597	total: 628ms	remaining: 2.51s
700:	learn: 0.6347301	total: 718ms	remaining: 2.36s
800:	learn: 0.6146036	total: 809ms	remaining: 2.22s
900:	learn: 0.5977558	total: 900ms	remaining: 2.1s
1000:	learn: 0.5838666	total: 992ms	remaining: 1.98s
1100:	learn: 0.5723061	total: 1.08s	remaining: 1.87s
1200:	learn: 0.5621090	total: 1.18s	remaining: 1.76s
1300:	learn: 0.5528004	total: 1.27s	remaining: 1.66s
1400:	learn: 0.5442964	total: 1.37s	remaining: 1.56s
1500:	learn: 0.5367540	total: 1.47s	remaining: 1.46s
1600:	learn: 0.52967